# 01 — A neural network from scratch in numpy

Welcome to the final chapter of the course. We open it the way we have opened every method — **by
hand, before any library** — and we go one step beyond chapter 11: we assemble, in one clean place,
the **complete machinery of a neural network**. Forward pass, loss, the gradient (backward pass), the
training loop, and an honest held-out evaluation — all in plain numpy, on a small problem we can see.

This notebook is our **reference build**. In notebook 7 we will hand the very same network to
**PyTorch** and watch it do the backward pass for us; in notebook 2 we will stack this network deeper.
Everything starts here.

**Prerequisites**
- Chapter 11 — the neuron and the hidden layer (NB 1–2), backpropagation as the chain rule (NB 3),
  the softmax output head (NB 4).
- Chapter 00 — the train/test split, accuracy, and feature scaling.

**What you'll be able to do**
- Write a network's forward pass, loss, and backward pass for a **multi-class** problem.
- Derive the one new piece — the **softmax + cross-entropy gradient** — and check it numerically.
- Train the network with gradient descent and read its loss curve.
- Evaluate it honestly on held-out data, and see where its errors fall.

## The machinery we are about to assemble

A neural network, trained, is six moving parts working together:

- the **forward pass** — turn inputs into predictions;
- the **loss** — measure how wrong those predictions are;
- the **backward pass** — the gradient of the loss with respect to every weight;
- the **training loop** (the **optimizer**) — nudge the weights downhill, over and over;
- the **train/test split** — learn on one set, judge on another;
- the **evaluation** — a single honest number on data the network never saw.

Almost all of this is already yours. Chapter 11 built a *binary* network by hand — the chain rule, the
gradient-descent loop — and chapter 00 built the split, scaling, and accuracy. Here we assemble a
clean, reusable, **multi-class** version. The **one genuinely new piece** is the gradient of the
**softmax + cross-entropy** output — and it turns out to wear a familiar face.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split

from ml_course import colors, viz

viz.use_course_style()

SEED = 0
np.random.seed(SEED)

X, y = make_blobs(
    n_samples=300, centers=3, n_features=2, cluster_std=1.2, random_state=SEED
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)

# Standardize using statistics from the TRAINING set only (no leakage) — chapter 00.
mu, sd = X_train.mean(axis=0), X_train.std(axis=0)
X_train_s = (X_train - mu) / sd
X_test_s = (X_test - mu) / sd

print("X:", X.shape, " class counts:", np.bincount(y))
print("train:", X_train_s.shape, " test:", X_test_s.shape)

## A problem we can see

We use three Gaussian blobs in two dimensions: **3 classes, 2 features**. Two features means we can
plot everything — the data, the decision regions, the errors. The blobs overlap a little where they
meet, so the problem is honest: no model will get every point right.

Chapter 11's neuron produced a single probability — fine for two classes. With **three** classes we
need an output that produces **three** probabilities that sum to 1. That is the job of the **softmax
head** (chapter 11, NB 4), which we now wire in.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for k in range(3):
    mask = y_train == k
    ax.scatter(
        X_train_s[mask, 0],
        X_train_s[mask, 1],
        color=colors.CLASS_CYCLE[k],
        edgecolor=colors.COLORS["text"],
        linewidth=0.6,
        s=45,
        label=f"class {k}",
    )
ax.set_xlabel("feature 1 (standardized)")
ax.set_ylabel("feature 2 (standardized)")
ax.set_title("Three classes, two features")
ax.legend()
plt.show()

**Read the figure.** Three clouds of points, one colour per class, on two standardized axes. They are
mostly separated, but look where two clouds touch — those overlap regions are where any classifier
will make its honest mistakes. Our job is to carve this plane into three sensible regions.

## The architecture: 2 → 16 → 3

Our network has three layers of values:

- an **input** of 2 numbers (the two features);
- a **hidden layer** of 16 neurons with a **ReLU** activation `max(0, z)` — what lets the network bend
  its boundaries (chapter 11, NB 2);
- an **output** of 3 numbers turned into probabilities by the **softmax**.

The softmax takes three raw scores (the *logits*) `z₀, z₁, z₂` and returns three probabilities:

`softmax(z)ₖ = e^(zₖ) / (e^(z₀) + e^(z₁) + e^(z₂))`

so each is in (0, 1) and the three sum to 1. The class with the largest probability is the prediction.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.axis("off")
C = colors.COLORS


def draw_layer(x, ys, color, labels=None):
    nodes = []
    for i, yv in enumerate(ys):
        ax.add_patch(plt.Circle((x, yv), 0.16, color=color, ec=C["text"], lw=1.1, zorder=3))
        if labels is not None:
            ax.text(x, yv, labels[i], ha="center", va="center", color=C["text"], zorder=4)
        nodes.append((x, yv))
    return nodes


inp = draw_layer(0.0, [1.6, 0.6], C["class_a"], ["x₁", "x₂"])
hid = draw_layer(2.0, [2.3, 1.7, 1.1, 0.5, -0.1], C["class_c"])
out = draw_layer(4.0, [1.7, 1.1, 0.5], C["class_b"])

arrow = dict(arrowstyle="->", color=C["grid"], lw=0.8)
for ix, iy in inp:
    for hx, hy in hid:
        ax.annotate("", xy=(hx - 0.16, hy), xytext=(ix + 0.16, iy), arrowprops=arrow)
for hx, hy in hid:
    for ox, oy in out:
        ax.annotate("", xy=(ox - 0.16, oy), xytext=(hx + 0.16, hy), arrowprops=arrow)

ax.text(0.0, -0.7, "input (2 features)", ha="center", color=C["text"])
ax.text(2.0, -0.9, "hidden (16 ReLU)", ha="center", color=C["text"])
ax.text(4.0, -0.5, "softmax (3 classes)", ha="center", color=C["text"])
ax.set_xlim(-0.9, 5.3)
ax.set_ylim(-1.2, 2.7)
ax.set_title("A 2 → 16 → 3 network (5 of the 16 hidden units shown)")
plt.show()

**Read the figure.** Values flow left to right. The two inputs connect to all sixteen hidden units
(only five are drawn, to keep the picture readable), and every hidden unit connects to the three
softmax outputs. Each arrow is a weight. Picturing all sixteen hidden units (only five are drawn), the
connections live in two weight matrices — **W₁** of shape
(2 × 16) from input to hidden, and **W₂** of shape (16 × 3) from hidden to output — plus a bias vector
for each layer (**b₁**, **b₂**). Those four arrays are everything the network learns.

## The forward pass

The forward pass is three steps, each a matrix multiply plus an activation:

- hidden pre-activation: `z₁ = X·W₁ + b₁`
- hidden activation: `h = ReLU(z₁) = max(0, z₁)`
- output: `logits = h·W₂ + b₂`, then `p = softmax(logits)`

`X` holds a whole batch of examples as rows, so each step processes all of them at once. We code it,
initialize the weights **small and random** — small so the untrained network starts undecided; random
so the hidden units do not all learn the same thing (the symmetry-breaking of chapter 11, NB 3) — and
read the loss before any training.

In [ ]:
def init_params(n_in=2, n_hidden=16, n_out=3, scale=0.1, seed=SEED):
    rng = np.random.default_rng(seed)
    return {
        "W1": rng.standard_normal((n_in, n_hidden)) * scale,
        "b1": np.zeros(n_hidden),
        "W2": rng.standard_normal((n_hidden, n_out)) * scale,
        "b2": np.zeros(n_out),
    }


def relu(z):
    return np.maximum(0.0, z)


def softmax(logits):
    z = logits - logits.max(axis=1, keepdims=True)  # subtract the row max for numerical stability
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def forward(params, X):
    z1 = X @ params["W1"] + params["b1"]
    h = relu(z1)
    logits = h @ params["W2"] + params["b2"]
    probs = softmax(logits)
    cache = {"X": X, "z1": z1, "h": h, "probs": probs}
    return probs, cache


def cross_entropy(probs, y):
    n = len(y)
    return float(-np.mean(np.log(probs[np.arange(n), y] + 1e-12)))


params = init_params()
probs_untrained, _ = forward(params, X_train_s)
print(f"untrained loss = {cross_entropy(probs_untrained, y_train):.4f}   (ln 3 = {np.log(3):.4f})")

## The loss: cross-entropy

For one example whose true class is `c`, the loss is `−log p_c`: the network is penalized by how small
a probability it gave the *correct* class. Across the batch we average:

`L = −(1/n) Σᵢ log p(i, yᵢ)`

This is the **cross-entropy** (chapter 03, NB 3 — now with more than two classes). Notice the number we
printed above: an untrained network spreads its probability roughly evenly over the 3 classes, so each
`p_c ≈ 1/3` and the loss sits near `−log(1/3) = ln 3 ≈ 1.10`. That is the "knowing nothing" baseline;
training has to drive it down.

## The backward pass: the one new derivation

Training needs the gradient of the loss with respect to every weight. We get it the same way as
chapter 11 — the **chain rule, layer by layer** — working backward from the loss.

The new piece is the very first step: the gradient of the **softmax + cross-entropy** at the logits.
Worked through — the **softmax Jacobian** and the `1/p_c` from the `−log p_c` cancel — it collapses to
something remarkably clean:

`∂L/∂logits = (p − y_onehot) / n`

where `y_onehot` puts a 1 in the true-class column. This is the **same shape** as the binary
`(p − y)` gradient from chapter 03 — predicted minus actual — now with one entry per class. A
genuinely new derivation, a genuinely familiar result.

From there, the rest is the chapter-11 chain rule, which we reuse without re-deriving:

- `∂L/∂W₂ = hᵀ · ∂L/∂logits`, and `∂L/∂b₂` sums those rows;
- propagate back through `W₂`:  `∂L/∂h = ∂L/∂logits · W₂ᵀ`;
- through the ReLU, which passes the gradient only where it was active:  `∂L/∂z₁ = ∂L/∂h ⊙ (z₁ > 0)`;
- finally  `∂L/∂W₁ = Xᵀ · ∂L/∂z₁`, and `∂L/∂b₁` sums the rows.

In [ ]:
def backward(params, cache, y):
    X, z1, h, probs = cache["X"], cache["z1"], cache["h"], cache["probs"]
    n = len(y)
    y_onehot = np.zeros_like(probs)
    y_onehot[np.arange(n), y] = 1.0

    d_logits = (probs - y_onehot) / n          # softmax + cross-entropy gradient: the one new piece
    grads = {"W2": h.T @ d_logits, "b2": d_logits.sum(axis=0)}
    dh = d_logits @ params["W2"].T
    dz1 = dh * (z1 > 0)                          # ReLU passes the gradient only where it was active
    grads["W1"] = X.T @ dz1
    grads["b1"] = dz1.sum(axis=0)
    return grads


def numerical_grad(params, X, y, key, idx, eps=1e-6):
    flat = params[key].ravel()
    original = flat[idx]
    flat[idx] = original + eps
    loss_plus = cross_entropy(forward(params, X)[0], y)
    flat[idx] = original - eps
    loss_minus = cross_entropy(forward(params, X)[0], y)
    flat[idx] = original
    return (loss_plus - loss_minus) / (2 * eps)


params = init_params()
_, cache = forward(params, X_train_s)
grads = backward(params, cache, y_train)

check_rng = np.random.default_rng(1)
max_rel_error = 0.0
for key in params:
    flat_grad = grads[key].ravel()
    for idx in check_rng.choice(flat_grad.size, size=min(10, flat_grad.size), replace=False):
        analytic = flat_grad[idx]
        numeric = numerical_grad(params, X_train_s, y_train, key, idx)
        denom = max(1e-12, abs(analytic) + abs(numeric))
        max_rel_error = max(max_rel_error, abs(analytic - numeric) / denom)
print(f"largest relative gradient error = {max_rel_error:.2e}")

**Read the result.** We compared our analytic gradient to a **finite-difference** estimate — nudge one
weight by a tiny ε, see how much the loss moves, and that ratio approximates the derivative. The
largest relative disagreement across the weights we checked is on the order of `1e-8`: our by-hand
gradient matches numerical truth. We can trust it to train with — the same gradient-check habit from
chapter 11, NB 3.

## Going further (optional): why the gradient is `(p − y)`

The clean form is worth seeing once. For a single example with true class `c`, the loss is `−log p_c`,
and the softmax is `p_k = e^(z_k) / Σⱼ e^(z_j)`. Two facts combine:

- the **softmax Jacobian**  `∂p_k/∂z_j = p_k · (δ_(kj) − p_j)`, where `δ_(kj)` is 1 when `k = j`, else 0;
- the loss touches the logits only through `p_c`, so by the chain rule
  `∂(−log p_c)/∂z_j = −(1/p_c) · ∂p_c/∂z_j = −(1/p_c) · p_c · (δ_(cj) − p_j) = p_j − δ_(cj)`.

That last expression is exactly `p − y_onehot` for this example — the `−1` lands in the true-class slot.
Averaging over the batch divides by `n`, giving `(p − y_onehot) / n`. The `1/p_c` from the log and the
`p_c` from the softmax **cancel** — which is *why* a forbidding-looking softmax gradient comes out as
predicted minus actual, the same shape as chapter 03's binary `(p − y)`.

## The training loop (the optimizer)

The optimizer is the gradient-descent loop you already know (chapter 03, NB 4; chapter 11, NB 3), now
driving four arrays instead of one. One **epoch** is one full pass over the training data; in each we:

1. run the **forward pass** to get predictions and the loss;
2. run the **backward pass** to get the gradients;
3. take a small step **downhill**:  `θ ← θ − η · ∂L/∂θ`  for every weight `θ`, with learning rate `η`.

We use the whole training set in each step (*full-batch* gradient descent), the simplest version.
Notebook 8 will meet *mini-batches*, where each step uses a small random slice instead.

In [ ]:
params = init_params()
learning_rate = 0.5
epochs = 400
losses = []

for _ in range(epochs):
    probs, cache = forward(params, X_train_s)
    losses.append(cross_entropy(probs, y_train))
    grads = backward(params, cache, y_train)
    for key in params:
        params[key] = params[key] - learning_rate * grads[key]

print(f"loss: {losses[0]:.4f} -> {losses[-1]:.4f}   over {epochs} epochs")

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(losses, color=colors.COLORS["model"], lw=2)
ax.axhline(np.log(3), color=colors.COLORS["muted"], ls="--", lw=1, label="ln 3 (random guessing)")
ax.set_xlabel("epoch")
ax.set_ylabel("cross-entropy loss")
ax.set_title("Training loss")
ax.legend()
plt.show()

**Read the figure.** The loss starts at the "knowing nothing" baseline `ln 3 ≈ 1.10` (dashed line) and
falls steeply as gradient descent tunes the weights, then flattens near `0.31` — the network has
settled into a good fit and further epochs change little. A falling, flattening loss curve is the
picture of training working.

## Evaluation: the honest number

A low *training* loss only says the network fit the data it saw. The question that matters is how it
does on data it never saw — the **held-out test set** (chapter 00). We measure accuracy (the fraction
classified correctly) on both, and draw the regions the network learned, so we can see *where* it
succeeds and fails.

In [ ]:
class NumpyNet:
    # Tiny wrapper so the by-hand network exposes .predict for viz.plot_decision_boundary.

    def __init__(self, params):
        self.params = params

    def predict(self, X):
        return forward(self.params, X)[0].argmax(axis=1)


net = NumpyNet(params)
train_acc = (net.predict(X_train_s) == y_train).mean()
test_acc = (net.predict(X_test_s) == y_test).mean()
print(f"train accuracy = {train_acc:.4f}")
print(f"test  accuracy = {test_acc:.4f}   (chance = {1 / 3:.4f})")

fig = viz.plot_decision_boundary(net, X_test_s, y_test)
fig.axes[0].set_title(f"Learned decision regions (test accuracy {test_acc:.3f})")
plt.show()

**Read the figure.** The network carved the plane into three regions, one per class, with gently
curved borders (the ReLU hidden layer at work). The test points mostly land in the right region; the
handful that do not sit right in the overlap zones we spotted at the start — genuinely ambiguous
points, where we would hesitate too. Training accuracy ≈ 0.87 and test accuracy ≈ 0.89 both sit far
above the 0.33 of random guessing, and within sampling noise of each other (the test set is only 75
points): the network **generalized** rather than memorized.

## What you built

You assembled a complete neural network from scratch in numpy:

- the **forward pass** — matrix multiplies, a ReLU hidden layer, a softmax head;
- the **cross-entropy loss**, with its `ln K` "knowing nothing" baseline;
- the **backward pass** — deriving the one new piece, the softmax + cross-entropy gradient
  `(p − y_onehot) / n`, and checking it against finite differences (`~1e-8`);
- the **training loop** — full-batch gradient descent driving the loss down;
- an **honest held-out evaluation**, with the decision regions drawn.

This exact network is our reference. In **notebook 7** we hand it to **PyTorch** and watch autograd
reproduce, automatically, the very backward pass you wrote. In **notebook 2** we stack it deeper
and ask what depth buys us. The only genuinely new mathematics here was the softmax gradient; every
other part you already owned.

## Your turn

1. **(warm-up)** Change the hidden size from 16 to 4, then to 64, and re-run. Does the test accuracy
   or the shape of the decision regions change much on this easy problem?
2. **(core)** Set the learning rate `η` to `0.05`, then to `2.0`, keeping the epochs fixed. What
   happens to the loss curve — too slow to converge, or unstable? Find a value you would trust.
3. **(core)** Raise the blobs' `cluster_std` from 1.2 to 2.0 and re-run the whole notebook. The classes
   now overlap more: how far does test accuracy fall, and *where* do the new errors land?
4. **(reach)** Add a second hidden layer (`2 → 16 → 16 → 3`). Extend the forward pass by one more
   `matmul + ReLU`, and the backward pass by one more chain-rule step. Does the deeper network do
   better on this easy problem? (That is a preview of notebook 2.)

## Where this goes next

You now own a complete network. Notebook 2 asks the question this chapter is built around: if one
hidden layer is good, what does **stacking many** of them buy — and what new trouble does depth bring?
That is where "deep" learning earns its name.

## References

- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*, chapter 6 (deep feedforward
  networks). MIT Press. https://www.deeplearningbook.org
- Chapter 11 — NB 3 (backpropagation as the chain rule), NB 4 (the softmax output head).
- Chapter 03 — NB 3 (cross-entropy / log-loss), NB 4 (gradient descent).

*Previous:* chapter 11 — the multi-layer perceptron.  *Next:* **12.2 — depth is a representation
hierarchy.**